# Pipeline E2 en Google Colab

Corre el pipeline completo (staging → limpieza → DW poblado → DataMart) contra BigQuery, sin instalar nada en tu computadora.

1. **Entorno de ejecución → Ejecutar todo** (o celda por celda).
2. Cuando lo pida, inicia sesión con la cuenta de Google que tiene acceso al proyecto `mineria-metro-cdmx`.
3. Al final se descarga `resultados_e2.zip`: descomprímelo en la raíz de tu clon del repositorio (sobrescribe `entregas/E2_eda_preparacion/tablas/` e `img/`).

Tiempo aproximado: 2 a 4 minutos de instalación + 5 a 10 minutos de pipeline.

In [ ]:
# Parámetros: rama del repositorio y proyecto de BigQuery
RAMA = "feat/e2"
PROYECTO = "mineria-metro-cdmx"

In [ ]:
# 1. Clonar el repositorio (copia limpia)
import os
os.chdir("/content")
!rm -rf Proyecto_Mineria_Datos
!git clone --depth 1 --branch {RAMA} https://github.com/Fernando-BM/Proyecto_Mineria_Datos.git
os.chdir("/content/Proyecto_Mineria_Datos")
!git log -1 --format="Commit %h · %ad · %s" --date=iso

In [ ]:
# 2. Instalar las versiones fijadas
!python --version
!pip install -q -r pipeline/requirements.txt

In [ ]:
# 3. Autenticación: abre una ventana para iniciar sesión con Google.
#    No se guarda ninguna llave en el repositorio.
from google.colab import auth
auth.authenticate_user()
os.environ["GCP_PROJECT"] = PROYECTO

In [ ]:
# 4. Prueba rápida sin BigQuery (muestra versionada en el repo): ~10 s
!python -W ignore pipeline/run_pipeline.py --fuente muestra

In [ ]:
# 5. Corrida oficial: lee metro_staging y puebla metro_dw y metro_mart
import time
t0 = time.time()
!python -W ignore pipeline/run_pipeline.py --fuente bigquery --destino bigquery
print(f"Tiempo total: {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 6. Evidencia: validaciones en BigQuery y resultados del DataMart
from IPython.display import Markdown, display
for nombre in ["validaciones_bigquery", "mart_p1_rollup_linea_dia", "mart_p4_anomalias_por_causa",
               "mart_p5_absorcion_cierre_l1"]:
    display(Markdown(f"### {nombre}\n" + open(f"entregas/E2_eda_preparacion/tablas/{nombre}.md").read()))

In [ ]:
# 7. Descargar los resultados para copiarlos al repositorio local
!zip -q -r resultados_e2.zip entregas/E2_eda_preparacion/tablas entregas/E2_eda_preparacion/img pipeline/salidas/ultima_ejecucion.log
from google.colab import files
files.download("resultados_e2.zip")